# SANA: учим сценариста комиксов · небольшой пилот

**До запуска ноутбука модель ещё не обучена.** Здесь дообучается текстовая
Qwen3-4B-Instruct-2507 (Apache-2.0): сценарии и описания кадров. Рисующая модель
не меняется. Пилот: 16 оригинальных сценариев для обучения + 4 отдельные темы для
сравнения до/после, русский и казахский, 4–6 классы. Это небольшой синтетический
набор, не доказательство того, что школьникам станет смешнее.

1. **Среда выполнения → Сменить тип среды → T4 GPU → Сохранить.**
2. **Среда выполнения → Выполнить все.** При первом запуске Google может спросить,
   разрешить ли выполнение кода из GitHub.
3. Дождись окончания и скачай `sana-comic-pilot.zip`. Открой `comparison.html`
   и сравни варианты A/B до просмотра ключа.

Платный тариф и секретные ключи не нужны. Бесплатный GPU может быть недоступен;
Colab может остановить сессию. Ноутбук не подключает Google Drive, не публикует
модель и не меняет сайт. Текущий Cloudflare endpoint не принимает этот адаптер;
публикация собственных весов — отдельная задача. Colab здесь не сервер сайта.

[Проверка рисунков и Neurons](https://learn-sana.pages.dev/examples/comic-page-review/)
· [Код и ограничения пилота](https://github.com/batyrkekw/SANA/tree/main/training/comics)


## 1. Проверяем облачный GPU
Если здесь ошибка про GPU, выбери T4 в меню среды выполнения. Покупать тариф не нужно.


In [ ]:
import shutil
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Нет NVIDIA GPU. Среда выполнения → Сменить тип среды → T4 GPU.")
gpu = torch.cuda.get_device_properties(0)
if gpu.total_memory < 14 * 1024**3:
    raise RuntimeError("Для пилота нужен GPU примерно с 15–16 ГБ памяти или больше.")
if shutil.disk_usage("/content").free < 16 * 1024**3:
    raise RuntimeError("Для загрузки модели нужно не меньше 16 ГБ свободного места в Colab.")
print("GPU:", gpu.name, "· память:", round(gpu.total_memory / 1024**3, 1), "ГБ")
print("Вычисления пойдут в Colab. GPU твоего компьютера не используется.")


## 2. Загружаем проверенную версию кода
Файлы взяты с конкретного коммита SANA и проверяются по SHA-256. Устанавливаются закреплённые версии библиотек; CUDA PyTorch из Colab сохраняется.


In [ ]:
import hashlib
import json
import subprocess
import sys
import urllib.request
from pathlib import Path

REVISION = '27a74be2e1f3aa59195f455ff046000fb58e2358'
EXPECTED_SHA256 = {'protocol.py': 'aa4a815c100e09bd5d53e625affcca1322d56ffa99d92567f37ffdf94b185da9', 'train.py': '8d420be0b9ac774eeded25592793b1a68ddbfc12af19b2ae0f40c0d38f711c72', 'check_pipeline.py': 'c7bbc6369934285a66c2840930cb364fa0969c4a220f24929943d914ebba2ccf', 'requirements-colab.txt': 'ae4ad08ba211bab98b13088eb3af8e155257d26426a5f2fe97a392f6385baa5d', 'data/stories.jsonl': '5428ac37a111fabc91ee390112e0525b19138df21bea3d9b8f93538006f30d58'}
SOURCE = Path("/content/sana-training/comics")
SOURCE.mkdir(parents=True, exist_ok=True)
for name, expected in EXPECTED_SHA256.items():
    url = f"https://raw.githubusercontent.com/batyrkekw/SANA/{REVISION}/training/comics/{name}"
    with urllib.request.urlopen(url, timeout=60) as response:
        content = response.read()
    if hashlib.sha256(content).hexdigest() != expected:
        raise RuntimeError(f"Контрольная сумма не совпала: {name}")
    target = SOURCE / name
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_bytes(content)
(SOURCE / "source-revision.txt").write_text(REVISION)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(SOURCE / "requirements-colab.txt")], check=True)
print("Код загружен:", REVISION)


## 3. Проверяем данные и разметку ответа
Оценочные темы не попадают в обучение. Ошибочная длина или разметка останавливает запуск. Пример ниже — авторский сценарий из обучающей части, не результат новой модели.


In [ ]:
subprocess.run([sys.executable, str(SOURCE / "train.py"), "--check-data"], check=True)
subprocess.run([sys.executable, str(SOURCE / "check_pipeline.py")], check=True)
records = [json.loads(line) for line in (SOURCE / "data/stories.jsonl").read_text().splitlines()]
preview = next(row for row in records if row["split"] == "train")
print("Пример:", preview["target"]["title"])
for i, panel in enumerate(preview["target"]["panels"], 1):
    print(f"{i}. {panel['dialogue']} — {panel['narration']}")


## 4. Настоящее GPU-обучение
Сначала модель ответит на четыре проверочных запроса без адаптера. Затем пройдут
две эпохи обучения LoRA и повтор тех же запросов. В логах появится `trainable params`
и прогресс по шагам. На T4 это может занять заметное время; точное время ещё не измерено.

Не перезапускай эту ячейку, пока она выполняется. Понижение loss не доказывает,
что истории стали смешнее: решающим будет сравнение текстов. Если память закончилась
или сессия оборвалась, сохрани текст ошибки; скрытого перехода на платный сервис нет.


In [ ]:
from datetime import datetime, timezone

RUN_DIR = Path("/content") / ("sana-comic-" + datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S"))
print("Результаты:", RUN_DIR, flush=True)
subprocess.run([
    sys.executable, "-u", str(SOURCE / "train.py"),
    "--output", str(RUN_DIR), "--epochs", "2", "--max-length", "2048",
], check=True)
manifest = json.loads((RUN_DIR / "manifest.json").read_text())
print("Статус:", manifest["status"])
print("Шагов обучения:", manifest["steps"])
print("Пик памяти GPU:", round(manifest["peak_gpu_gib"], 2), "ГБ")


## 5. Скачиваем и сравниваем
ZIP содержит адаптер, токенизатор, параметры запуска, результаты до/после и
`comparison.html`. Базовые веса в архив не входят. Скачай его до отключения Colab:
файлы временной среды могут пропасть.

Сначала оцени A/B по точности, понятности, сюжету, юмору и рисуемости (0–2 по каждому).
Фактическая ошибка отклоняет вариант независимо от суммы. Только после оценки
открой `comparison-key.json`. Эти ответы ещё не опубликованы на сайте.


In [ ]:
from google.colab import files
from IPython.display import HTML, display

archive = RUN_DIR / "sana-comic-pilot.zip"
if not archive.exists():
    raise RuntimeError("Архив ещё не создан. Проверь завершение предыдущей ячейки.")
files.download(str(archive))
display(HTML((RUN_DIR / "comparison.html").read_text()))


## Что дальше
Сохрани архив и результаты сравнения. Если адаптер не улучшил истории, его не
следует подключать к продукту. Для более сильного обучения понадобится больше
проверенных сценариев, обратная связь учителей и школьников, отдельная оценка
казахского языка. Обучение иллюстратора требует исправленных визуальных образцов;
рисунки с ошибками из первого теста сюда не включены.

Источники: [Qwen и лицензия](https://huggingface.co/Qwen/Qwen3-4B-Instruct-2507),
[QLoRA/PEFT](https://huggingface.co/docs/peft/developer_guides/quantization),
[ограничения Colab](https://research.google.com/colaboratory/faq.html),
[LoRA в Cloudflare](https://developers.cloudflare.com/workers-ai/features/fine-tunes/loras/).
